In [20]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# **Tokenizer**

In [2]:
vocab = {
    "the": 0,
    "capital": 1,
    "of": 2,
    "united": 3,
    "state": 4,
    "is": 5,
    "not": 6,
    "london": 7,
    "france": 8,
    "paris": 9,
    "and": 10,
    "berlin": 11,
    "germany": 12,
    "rome": 13,
    "in": 14,
    "italy": 15,
    "madrid": 16,
    "spain": 17,
    "lisbon": 18,
    "portugal": 19,
    "kingdom": 20,
    "washington": 21,
    "although": 22,
    "these": 23,
    "place": 24,
    "are": 25,
    "often": 26,
    "mention": 27,
    "together": 28,
    "each": 29,
    "country": 30,
    "has": 31,
    "its": 32,
    "own": 33,
    "identity": 34,
    "any": 35,
    "european": 36,
    "city": 37,
    "remain": 38,
    "important": 39,
    "with": 40,
    "a": 41,
    "rich": 42,
    "history": 43,
    "culture": 44,
    "europe": 45,
    "made": 46,
    "many": 47,
    "unique": 48,
    "world": 49,
    "while": 50,
    "known": 51,
    "for": 52,
    "art": 53,
    "fashion": 54,
    "famous": 55,
    "they": 56,
    "ed": 57,
    "s": 58,
    ".": 59,
    ",": 60,
    " ": 61,
    "<unk>": 62,
    "<pad>": 63
  }



In [3]:
class UstaTokenizer:
  def __init__(self, vocab):
      self.vocab = vocab
      self.reverse_vocab = {v: k for k, v in self.vocab.items()}

  def encode(self, text):
    tokens = []

    for word in text.split():
      i = 0
      # example: states
      # state => 4
      # s => 58
      while i < len(word):
        found_match = False
        for j in range(len(word), i, -1):
          sub_word = word[i:j]
          if sub_word in self.vocab:
            tokens.append(self.vocab[sub_word])
            i = j
            found_match = True
            break
        if not found_match:
          tokens.append(self.vocab["<unk>"])
          i += 1
      tokens.append(self.vocab[" "])

    # check if text is not ends with a space
    if not text.endswith(" "):
      tokens.pop()
    return torch.tensor(tokens)

  def tokenize(self, text):
    token_ids = self.encode(text)
    # token_ids from tensor to list
    token_ids = token_ids.detach().numpy().tolist()

    return [self.reverse_vocab[id] for id in token_ids]

  def decode(self, ids):
    text = ""
    for id in ids:
      text += self.reverse_vocab[id]
    return text



In [4]:

u_tokenizer = UstaTokenizer(vocab)

prompt = "the capital of the united"

tokens = u_tokenizer.encode(prompt)
tokens

tensor([ 0, 61,  1, 61,  2, 61,  0, 61,  3])

# **Text**

In [5]:
text = """the capital of the united states is not london. the capital of france is paris, and berlin is the capital of germany. rome is in italy,

madrid is in spain, and lisbon is in portugal. the capital of the united kingdom is not paris, and the capital of the united states is not berlin.
although these places are often mentioned together, although these capitals are often mentioned together, although these are often mentioned together,
each country has its own capital, and each country has its own city, and each capital has its own identity, and each capital has its own history. washington
is the capital of the united states, and london is the capital of the united kingdom. paris is known for art and fashion, and berlin is known for art and
history, and rome is known for art and history, and madrid is known for culture and history, and lisbon is known for culture and art. rome is rich with culture,
rome is rich with history, rome is rich with art, and madrid is rich with art and culture. lisbon is a unique city in portugal with a rich history, a rich culture,
and a rich identity. these capitals are often mentioned together, these capitals are often mentioned together in art, these capitals are often mentioned together
in culture, these capitals are often mentioned together in history. the united states is not in europe, the united states is not in any european place, and
washington is not in any european city. each european country is made of important capitals, and each european capital is made of art, history, and culture.
the capital of the united states is washington, the capital of the united kingdom is london, the capital of france is paris, the capital of germany is berlin,
the capital of italy is rome, the capital of spain is madrid, and the capital of portugal is lisbon. while these capitals are in europe, while these capitals are
in europe, washington is in the united states. these capitals remain important, these remain important, these places remain important in the world. the
capital of each country is its own, the capital of each country is its identity, the capital of each country is its culture. europe is made of many,
europe is made of many capitals, europe is made of many important places. each place is rich with culture, each place is rich with history, and each capital is

rich with identity. the world is made of capitals, the world is made of, the world is made of places, and the capital of the united states is washington,
not any european city, not paris, not london, not berlin. the capital of the united states is not london. the capital of france is paris, and berlin is the capital of germany.
rome is in italy, madrid is in spain, and lisbon is in portugal. the capital of the united kingdom is not paris, and the capital of the united
states is not berlin. although these places are often mentioned together, each country has its own capital, and each capital has its own identity.
washington is the capital of the united states, and london is the capital of the united kingdom. paris is known for art and fashion, while berlin is
famous for its culture and history. rome is rich with history, and madrid is known for its art and culture. lisbon is a unique city in portugal
with a rich history. these capitals are often mentioned together, although each place with its own culture. the united states is not in europe,
and washington is not in any european country. these european capitals are made of history, culture, and identity. each country in europe has a capital,
and each capital is known for important. london, paris, berlin, rome, madrid, and lisbon remain important places in the world. while these capitals
are in europe, washington is in the united states. although these places are not in the country, they are often mentioned together in art, culture,
and history. the capital of each country is its own. europe is made of many capitals, and each has a capital with a unique history.
the world is of important places, and the capital of the united states is washington, not any european city. """

# **Model**

# **Embedding**

In [6]:
import torch
import torch.nn as nn


def get_rotary_position_encoding(input: torch.Tensor, base=10000, device="cpu"):
  context_length, dimension = input.shape

  assert dimension % 2 == 0, "dimension must be even"

  half_dimension = dimension // 2

  freqs_indices = torch.arange(0, half_dimension, device=device, dtype=torch.float32)

  freqs = 1.0 / (base ** (freqs_indices / dimension))

  positions = torch.arange(0, context_length, device=device, dtype=torch.float32).unsqueeze(1)

  angles = positions * freqs

  sin_angles = torch.sin(angles)
  cos_angles = torch.cos(angles)

  input_even = input[:, :dimension // 2] # [0, 2, 4, ..]
  input_odd = input[:, dimension // 2:] # [1, 3, 5, ..]

  input_even_rotated = input_even * cos_angles - input_odd * sin_angles
  input_odd_rotated = input_even * sin_angles + input_odd * cos_angles

  input_rotated = torch.empty_like(input)

  input_rotated[:, :dimension // 2] = input_even_rotated
  input_rotated[:, dimension // 2:] = input_odd_rotated

  return input_rotated

class UstaEmbedding(nn.Module):
  def __init__(self, vocab_size, embedding_dim, context_length):
    super().__init__()
    # position embedding but not being used in the forward pass
    # it is just for educational purposes
    # self.pos_embedding = nn.Embedding(context_length, embedding_dim)
    # self.get_pos = get_rotary_position_encoding
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.get_pos = get_rotary_position_encoding

  def forward(self, x):
    x = self.embedding(x)
    x = self.get_pos(x)
    return x



# Multi Head Attention

In [7]:
class UstaMultiHeadAttention(nn.Module):
  def __init__(self, embedding_dim, output_dim, context_length, num_heads, dropout_rate = 0):
    super().__init__()

    self.context_length = context_length

    self.multi_head_attention = nn.MultiheadAttention(embedding_dim, num_heads, dropout=dropout_rate)
    self.projection = nn.Linear(embedding_dim, output_dim)

    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

  def forward(self, x):
    number_of_tokens = x.shape[0]
    x = x[:self.context_length]
    attention_mask = self.mask[:number_of_tokens, :number_of_tokens]
    out, _ = self.multi_head_attention(x, x, x, attn_mask=attention_mask)
    out = self.projection(out)
    return out

# **Layer Normalization**

In [8]:
class UstaLayerNorm(nn.Module):
  def __init__(self, embedding_dim, eps=1e-5):
    super().__init__()
    self.eps = eps

    self.weight = nn.Parameter(torch.ones(embedding_dim))


  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    variance = x.var(dim=-1, keepdim=True, unbiased=False)
    normalized_x = (x - mean) / torch.sqrt(variance + self.eps)
    return self.weight * normalized_x

# **MLP BLOCK**

In [9]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (
      1 + torch.tanh(
          torch.sqrt(torch.tensor(2 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
        )
    )

class UstaMLP(nn.Module):
  def __init__(self, embedding_dim, hidden_dim):
    super().__init__()

    self.gate_proj = nn.Linear(embedding_dim, hidden_dim)
    self.up_proj = nn.Linear(embedding_dim, hidden_dim)
    self.down_proj = nn.Linear(hidden_dim, embedding_dim)
    self.gelu = GELU()

  def forward(self, x):
    """ gate = self.gate_proj(x)
        gate = F.gelu(gate, approximate="tanh")
        up = self.up_proj(x)
        fuse = gate * up
        outputs = self.down_proj(fuse) """
    gate = self.gate_proj(x)
    gate = self.gelu(gate)
    up = self.up_proj(x)
    fuse = gate * up
    outputs = self.down_proj(fuse)
    return outputs

# **Decoder Block**

In [10]:
class UstaDecoderBlock(nn.Module):
  def __init__(self, embedding_dim, num_heads, context_length):
    super().__init__()

    self.self_attention = UstaMultiHeadAttention(embedding_dim, embedding_dim, context_length, num_heads, dropout_rate=0.5)
    self.norm1 = UstaLayerNorm(embedding_dim)
    self.mlp = UstaMLP(embedding_dim, embedding_dim)
    self.norm2 = UstaLayerNorm(embedding_dim)

  def forward(self, x):
    res = self.norm1(x)

    x = self.self_attention(x)
    x = self.norm1(x)

    x = x + res

    res = self.norm2(x)
    x = self.mlp(x)
    x = self.norm2(x)

    x = x + res

    return x

# **Model Construction**

In [15]:
class UstaModel(nn.Module):
  def __init__(self, vocab_size, embedding_dim, num_heads, context_length, num_layers):
    super().__init__()

    self.embedding = UstaEmbedding(vocab_size, embedding_dim, context_length)
    self.layers = nn.Sequential(
      *[UstaDecoderBlock(embedding_dim, num_heads, context_length) for _ in range(num_layers)]
    )
    self.lm_head = nn.Linear(embedding_dim, vocab_size)


  def forward(self, x):
      x = self.embedding(x)  # dictionary meaning of the tokens (words)
      x = self.layers(x)
      x = self.lm_head(x)
      return x

# Model Training

In [16]:
context_length = 32

In [17]:
torch.manual_seed(1)
u_model = UstaModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=12, num_heads=4, context_length=context_length, num_layers=8)

out = u_model(tokens)
out.shape

torch.Size([9, 64])

In [18]:
token_ids = u_tokenizer.encode(text)
len(token_ids), type(token_ids)

(1594, torch.Tensor)

In [19]:
ids = token_ids.detach().cpu().numpy().tolist()
len(ids), type(ids)

(1594, list)

# **Text Dataset**

In [21]:
pad_id = 63

class TextDataset(Dataset):
  def __init__(self, token_ids: list, context_length: int, stride: int):
    super().__init__()

    self.inputs = []
    self.targets = []

    for i in range(0, len(token_ids) - context_length, stride):
      input_chunk = token_ids[i:i + context_length]
      target_chunk = token_ids[i + 1:i + context_length + 1]

      # truncate to context length
      input_chunk = input_chunk[:context_length]
      target_chunk = target_chunk[:context_length]

      # pad to context length
      input_chunk = input_chunk + [pad_id] * (context_length - len(input_chunk))
      target_chunk = target_chunk + [pad_id] * (context_length - len(target_chunk))

      # truncate to context length
      input_chunk = input_chunk[:context_length]
      target_chunk = target_chunk[:context_length]

      self.inputs.append(torch.tensor(input_chunk, dtype=torch.long))
      self.targets.append(torch.tensor(target_chunk, dtype=torch.long))

  def __len__(self):
    return len(self.inputs)

  def __getitem__(self, idx):
    return self.inputs[idx], self.targets[idx]


def create_data_loader(token_ids: list, context_length: int, stride: int,
                       batch_size: int, shuffle: bool = True, device: str = "cpu"):
  dataset = TextDataset(token_ids, context_length, stride)
  dataloader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=shuffle,
      generator=torch.Generator(device=device)
    )

  return dataloader

In [22]:
stride = 12

dataset = TextDataset(ids, context_length, stride)

len(dataset.inputs), len(dataset.targets)

(131, 131)

In [23]:
# model parameters count
parameters_count = sum(p.numel() for p in u_model.parameters())
print(parameters_count)

# model architecture
print(u_model)

11776
UstaModel(
  (embedding): UstaEmbedding(
    (embedding): Embedding(64, 12)
  )
  (layers): Sequential(
    (0): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (norm1): UstaLayerNorm()
      (mlp): UstaMLP(
        (gate_proj): Linear(in_features=12, out_features=12, bias=True)
        (up_proj): Linear(in_features=12, out_features=12, bias=True)
        (down_proj): Linear(in_features=12, out_features=12, bias=True)
        (gelu): GELU()
      )
      (norm2): UstaLayerNorm()
    )
    (1): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
    

In [24]:
out0 = u_model(dataset.inputs[0])
out0.shape

torch.Size([32, 64])

# **Loss Function**

In [25]:
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(out0, dataset.targets[0])
loss

tensor(4.3883, grad_fn=<NllLossBackward0>)

## **Optimizer**

In [26]:
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
optimizer = torch.optim.AdamW(u_model.parameters(), lr=1e-3)

In [27]:
for input, target in dataset:
  print(input.shape, target.shape)
  break

torch.Size([32]) torch.Size([32])


In [28]:
epoch = 10

for epoch in range(epoch):
  total_loss = 0.
  for input, target in dataset:
    pred = u_model(input)

    loss = loss_fn(pred, target)
    total_loss += loss.item()

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  average_loss = total_loss / len(dataset)
  print(f"Epoch {epoch + 1} loss: {loss.item()} average loss: {average_loss}")

Epoch 1 loss: 2.7942237854003906 average loss: 3.0079858594268334
Epoch 2 loss: 2.307847738265991 average loss: 2.466614244548419
Epoch 3 loss: 2.179509162902832 average loss: 2.229976992570717
Epoch 4 loss: 2.0944082736968994 average loss: 2.143867012198645
Epoch 5 loss: 2.1119813919067383 average loss: 2.1032867868438023
Epoch 6 loss: 2.121394395828247 average loss: 2.088291424831361
Epoch 7 loss: 2.1278445720672607 average loss: 2.072666466691112
Epoch 8 loss: 2.129866600036621 average loss: 2.080221701214332
Epoch 9 loss: 2.1160717010498047 average loss: 2.0760294062490683
Epoch 10 loss: 2.107538938522339 average loss: 2.0674220014164466


In [29]:
new_tokens = u_tokenizer.encode("madrid is in")
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(61)

out = u_model(torch.tensor(new_tokens))

probs = torch.softmax(out[-1], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

(tensor(0.0831, grad_fn=<MaxBackward0>),
 tensor(5),
 tensor([0.0594, 0.0598, 0.0478, 0.0268, 0.0201, 0.0831, 0.0224, 0.0102, 0.0041,
         0.0141, 0.0579, 0.0119, 0.0047, 0.0129, 0.0401, 0.0051, 0.0117, 0.0048,
         0.0106, 0.0061, 0.0086, 0.0131, 0.0094, 0.0219, 0.0166, 0.0235, 0.0145,
         0.0145, 0.0132, 0.0288, 0.0162, 0.0130, 0.0183, 0.0125, 0.0092, 0.0065,
         0.0119, 0.0086, 0.0064, 0.0125, 0.0178, 0.0129, 0.0168, 0.0246, 0.0186,
         0.0185, 0.0152, 0.0063, 0.0049, 0.0097, 0.0064, 0.0092, 0.0130, 0.0155,
         0.0035, 0.0023, 0.0017, 0.0018, 0.0019, 0.0011, 0.0028, 0.0022, 0.0002,
         0.0004], grad_fn=<SoftmaxBackward0>))

In [30]:
# save model
torch.save(u_model.state_dict(), "u_model.pth")

# load model
u_model.load_state_dict(torch.load("u_model.pth"))

<All keys matched successfully>

In [31]:
loaded_model = UstaModel(64, embedding_dim=12, num_heads=4, context_length=32, num_layers=8)
loaded_model.load_state_dict(torch.load("u_model.pth"))
loaded_model

UstaModel(
  (embedding): UstaEmbedding(
    (embedding): Embedding(64, 12)
  )
  (layers): Sequential(
    (0): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (norm1): UstaLayerNorm()
      (mlp): UstaMLP(
        (gate_proj): Linear(in_features=12, out_features=12, bias=True)
        (up_proj): Linear(in_features=12, out_features=12, bias=True)
        (down_proj): Linear(in_features=12, out_features=12, bias=True)
        (gelu): GELU()
      )
      (norm2): UstaLayerNorm()
    )
    (1): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (p